# Team 3: Synthetic Passkey Retrieval & Non-Inferiority Testing
**Measuring Associative Retrieval Collapse & Non-Inferiority Rejection ($\delta = 0.05$)**

This notebook evaluates synthetic Needle-In-A-Haystack retrieval with 16 orthonormal classes and executes bootstrap non-inferiority testing.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df_ret = pd.read_csv('../data/retrieval_raw.csv')
df_noninf = pd.read_csv('../data/noninferiority.csv')
print('Loaded retrieval runs:', len(df_ret))
df_noninf.head(10)

## 1. Retrieval Accuracy vs Sequence Length N
Comparison of Global Target retrieval accuracy across Softmax, Linear Attention, and SDPA.

In [ ]:
global_df = df_noninf[df_noninf.mode == 'global target']
plt.figure(figsize=(9, 5))
for method, grp in global_df.groupby('method'):
    # acc = 1.0 + delta (since softmax baseline is 1.0)
    acc = 1.0 + grp['delta']
    plt.plot(grp.N, acc, marker='o', lw=2, label=method)
plt.axhline(0.0625, color='gray', linestyle=':', label='Chance Level (1/16 = 6.25%)')
plt.xscale('log')
plt.xlabel('Sequence Length N (log scale)')
plt.ylabel('Passkey Retrieval Accuracy')
plt.title('Passkey Retrieval Accuracy: Softmax vs Linear Attention Collapse')
plt.grid(True, alpha=0.4)
plt.legend()
plt.tight_layout()
plt.show()

## 2. Bootstrap Non-Inferiority Analysis ($\Delta$ vs $\delta = 0.05$)
Rejection criteria: If upper confidence bound $CI_{hi} < -0.05$, non-inferiority is conclusively rejected.

In [ ]:
lin_global = global_df[global_df.method.str.contains('linear')]
print('Linear Attention Non-Inferiority Status:')
for _, row in lin_global.iterrows():
    print(f"N = {int(row.N):5d} | Delta: {row.delta:.3f} | 95% CI: [{row.ci_lo:.3f}, {row.ci_hi:.3f}] | Non-Inferior: {row.non_inferior}")